# PoC-HNSRM Framework Overview

The PoC-HNSRM (Proof of Concept - Health Nostics and System Reliability Monitor) framework integrates advanced machine learning models for prognostics and health management (PHM) of aerospace systems. This demonstration focuses on the integration of CNN-LSTM architectures with the C-MAPSS FD001 dataset for fault detection and remaining useful life prediction.

## Architecture Comparison: RandomForest/LightGBM vs CNN-LSTM

| Feature | RandomForest/LightGBM | CNN-LSTM |
|---------|----------------------|----------|
| **Input Type** | Tabular features | Sequential time-series (raw sensors) |
| **Feature Engineering** | Manual feature extraction required | Automatic feature learning via convolutions |
| **Temporal Dependencies** | Limited (via engineered lags) | Explicitly modeled via LSTM gates |
| **Spatial Patterns** | Not captured | Captured via convolutional filters |
| **Performance on FD001** | Good with careful feature engineering | End-to-end learning potential |


## CNN-LSTM Model Architecture

The CNN-LSTM architecture combines convolutional and recurrent layers for multivariate time-series analysis:

1. **Convolutional Block**: Extracts local patterns and reduces dimensionality from raw sensor inputs using 1D convolutions with ReLU activation and max pooling.
2. **LSTM Layer**: Processes the extracted feature sequences, capturing long-term temporal dependencies and temporal dynamics.
3. **Dense Output Layer**: Fully connected layer for classification (fault type) or regression (remaining useful life).

The model takes sliding windows of sensor readings as input and outputs fault classification or prognostic estimates.

In [ ]:
# Create CNN-LSTM model
model = create_cnn_lstm_model(
    input_shape=(window_size, n_sensors),
    lstm_units=128,
    num_classes=4  # Operating condition classes for FD001
)
model.summary()

## KerasSklearnAdapter Description

The `KerasSklearnAdapter` bridges Keras models with scikit-learn compatible workflows. It allows the CNN-LSTM model to be:
- Integrated into `GridSearchCV` for hyperparameter tuning
- Used with `Pipeline` for end-to-end preprocessing and modeling
- Evaluated using standard scikit-learn metrics and cross-validation

The adapter wraps the Keras model, handling `fit`, `predict`, and `score` methods compatible with scikit-learn conventions.

In [ ]:
# Create KerasSklearnAdapter instance
from composio_hnsr import KerasSklearnAdapter

adapter = KerasSklearnAdapter(
    model=model,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)

## LightClassifier with CNN-LSTM Description

The `LightClassifier` provides a lightweight, high-performance classification interface for deep learning models. When configured with `cnn_lstm` architecture, it:
- Instantiates the CNN-LSTM model with optimized hyperparameters
- Supports fast prediction on large-scale sensor datasets
- Integrates with the PoC-HNSRM framework for real-time inference
- Provides built-in methods for feature importance and interpretability via attention maps

The classifier is designed for deployment efficiency while maintaining the predictive power of the CNN-LSTM architecture.

In [ ]:
# Create and use LightClassifier with cnn_lstm architecture
from hnsr_light import LightClassifier

classifier = LightClassifier(architecture='cnn_lstm', input_shape=(window_size, n_sensors))
classifier.fit(X_train, y_train, epochs=50, batch_size=32)
y_pred = classifier.predict(X_test)
print("Accuracy:", classifier.score(X_test, y_test))